<a href="https://colab.research.google.com/github/saskiaalifah/SaskiaAlifah_2411531002_BigData26/blob/main/Praktikum6/Praktikum6/BD_P06_2411531002_SaskiaAlifah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 6 Big Data
# Apache Spark dengan PySpark





## D. Persiapan Praktikum

**Tujuan:** Menyiapkan lingkungan Apache Spark pada Google Colab tanpa instalasi manual.

### Langkah 3: Instal PySpark

In [1]:
!pip install pyspark

> **Interpretasi**
>
> - Perintah ini memasang library PySpark menggunakan `pip`. Tanda `!` menandakan perintah shell pada Colab.
> - Apabila PySpark sudah terpasang, akan muncul keterangan `Requirement already satisfied`.

### Langkah 4: Import Library dan Inisialisasi SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PraktikumBigData") \
    .master("local[*]") \
    .getOrCreate()

print("Versi Spark:", spark.version)

Versi Spark: 4.0.5


> **Interpretasi**
>
> - SparkSession berhasil dibuat, ditandai dengan tercetaknya versi Spark.
> - Nomor versi dapat berbeda bergantung pada versi PySpark yang terpasang.

### Langkah 5: Periksa Konfigurasi Spark

In [3]:
print("Spark Application Name:",
      spark.sparkContext.appName)

print("Spark Master:",
      spark.sparkContext.master)

Spark Application Name: PraktikumBigData
Spark Master: local[*]


> **Interpretasi**
>
> - **Application Name:** `PraktikumBigData`, sesuai dengan nilai pada `appName()`.
> - **Spark Master:** `local[*]`, yang berarti Spark berjalan secara lokal pada satu mesin dengan memanfaatkan seluruh core CPU.




## E.1. Mengenal RDD pada Apache Spark

**Tujuan:** Membuat RDD dan menerapkan operasi *transformation* serta *action*.

### Langkah 1: Membuat RDD

In [4]:
data = [10, 20, 30, 40, 50,
        60, 70, 80, 90, 100]

rdd = spark.sparkContext.parallelize(data)

print("Jumlah elemen:", rdd.count())

Jumlah elemen: 10


> **Interpretasi**
>
> - `parallelize()` mengubah list Python menjadi **RDD**.
> - `count()` merupakan **action** sehingga menghasilkan **10 elemen**, sesuai dengan isi list.

### Langkah 2: Melihat Isi RDD

In [5]:
print(rdd.collect())

[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


> **Interpretasi**
>
> - `collect()` mengambil semua elemen RDD ke driver sebagai list Python.
> - Hasil: `[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]` (sama dengan list awal).

### Pertanyaan Langkah 2

**1. Apa fungsi `collect()`? Mengapa termasuk action?**

**Jawaban:** `collect()` mengumpulkan seluruh isi RDD dari para worker ke driver. Ia termasuk **action** karena **memicu eksekusi** transformation yang tertunda dan **mengembalikan nilai** (bukan RDD baru).

### Langkah 3: Transformation `map()`
Kalikan setiap elemen dengan 2.

In [6]:
rdd_hasil = rdd.map(lambda x: x * 2)

print(rdd_hasil.collect())

[20, 40, 60, 80, 100, 120, 140, 160, 180, 200]


> **Interpretasi**
>
> - `map()` menerapkan operasi `x * 2` pada **setiap** elemen dan menghasilkan RDD baru dengan jumlah elemen yang sama.
> - Hasil: `[20, 40, 60, 80, 100, 120, 140, 160, 180, 200]`.

### Langkah 4: Transformation `filter()`
Ambil elemen yang lebih besar dari 50.

In [7]:
rdd_filter = rdd.filter(lambda x: x > 50)

print(rdd_filter.collect())

[60, 70, 80, 90, 100]


> **Interpretasi**
>
> - `filter()` hanya meloloskan elemen yang memenuhi `x > 50`.
> - Hasil: `[60, 70, 80, 90, 100]` (5 elemen). Angka 50 tidak ikut karena kondisinya `>`, bukan `>=`.

### Langkah 5: Menggabungkan Beberapa Transformation
Ambil bilangan genap, kemudian kalikan setiap bilangan dengan 3.

In [8]:
rdd_genap = rdd.filter(lambda x: x % 2 == 0)

rdd_kali = rdd_genap.map(lambda x: x * 3)

print(rdd_kali.collect())

[30, 60, 90, 120, 150, 180, 210, 240, 270, 300]


> **Interpretasi**
>
> - Dua transformation dirangkai secara berurutan: `filter()` (bilangan genap), kemudian `map()` (dikalikan 3).
> - Seluruh data merupakan kelipatan 10 (bilangan genap), sehingga semuanya lolos filter.
> - Hasil: `[30, 60, 90, 120, 150, 180, 210, 240, 270, 300]`.

### Tambahan (opsional): Bukti Lazy Evaluation pada RDD


In [9]:
# Tanpa action, hanya objek RDD yang tampil, bukan datanya
print(rdd_kali)

PythonRDD[4] at collect at /tmp/ipykernel_4033/1055199951.py:5


> **Interpretasi**
>
> - Yang tampil hanya referensi objek (`PythonRDD[...]`), **bukan data**.
> - Hal ini menunjukkan bahwa tanpa action, Spark belum memproses data.

### Pertanyaan E.1

**1. Apa perbedaan transformation dan action?**

**Jawaban:** **Transformation** (`map`, `filter`) menghasilkan RDD baru dan bersifat *lazy*. **Action** (`collect`, `count`) memicu eksekusi dan mengembalikan hasil ke driver.

**2. Mengapa `map()` dan `filter()` tidak langsung mengeksekusi pemrosesan data?**

**Jawaban:** Karena keduanya bersifat *lazy*: Spark hanya mencatat langkahnya (*lineage*/DAG). Dengan menunda eksekusi, Spark dapat mengoptimalkan seluruh rangkaian operasi sekaligus.

**3. Apa yang terjadi jika `collect()` tidak dipanggil?**

**Jawaban:** Tidak ada proses perhitungan dan tidak ada hasil yang ditampilkan. Yang ada hanya objek RDD berisi rencana transformation"

**4. Bagaimana jika data sangat besar dan seluruh hasil `collect()` dikirim ke driver?**

**Jawaban:** Seluruh data ditarik ke memori satu mesin sehingga berisiko menyebabkan **memori driver penuh** (`OutOfMemoryError`), **transfer jaringan yang lambat**, dan manfaat pemrosesan terdistribusi hilang. Alternatifnya, gunakan `take(n)`, `count()`, `reduce()`, atau simpan hasil ke penyimpanan terdistribusi.



## E.2. Membuat dan Mengolah DataFrame

**Tujuan:** Membuat DataFrame dan melakukan operasi dasar pengolahan data (dataset transaksi penjualan).

### Langkah 1: Membuat Dataset

In [10]:
data_transaksi = [
    (1, "Laptop", "Elektronik", 7500000, 2),
    (2, "Mouse", "Aksesoris", 150000, 5),
    (3, "Keyboard", "Aksesoris", 300000, 3),
    (4, "Monitor", "Elektronik", 2500000, 2),
    (5, "Flashdisk", "Aksesoris", 80000, 10),
    (6, "Printer", "Elektronik", 1800000, 1),
    (7, "Headset", "Aksesoris", 250000, 4),
    (8, "Laptop", "Elektronik", 7500000, 1),
    (9, "Mouse", "Aksesoris", 150000, 8),
    (10, "Monitor", "Elektronik", 2500000, 3)
]

kolom = [
    "id_transaksi",
    "produk",
    "kategori",
    "harga",
    "jumlah"
]

df = spark.createDataFrame(data_transaksi, kolom)

> **Interpretasi**
>
> - Dataset terdiri atas **10 transaksi** dengan 5 kolom.
> - `createDataFrame()` mengubah list berisi tuple menjadi DataFrame. Tidak terdapat output karena sel ini hanya membuat DataFrame.

### Langkah 2: Menampilkan Data

In [11]:
df.show()

+------------+---------+----------+-------+------+
|id_transaksi|   produk|  kategori|  harga|jumlah|
+------------+---------+----------+-------+------+
|           1|   Laptop|Elektronik|7500000|     2|
|           2|    Mouse| Aksesoris| 150000|     5|
|           3| Keyboard| Aksesoris| 300000|     3|
|           4|  Monitor|Elektronik|2500000|     2|
|           5|Flashdisk| Aksesoris|  80000|    10|
|           6|  Printer|Elektronik|1800000|     1|
|           7|  Headset| Aksesoris| 250000|     4|
|           8|   Laptop|Elektronik|7500000|     1|
|           9|    Mouse| Aksesoris| 150000|     8|
|          10|  Monitor|Elektronik|2500000|     3|
+------------+---------+----------+-------+------+



> **Interpretasi**
>
> - `show()` menampilkan seluruh **10 transaksi** dalam bentuk tabel (default maksimal 20 baris).
> - Produk yang sama dapat muncul pada beberapa transaksi: Laptop (id 1, 8), Mouse (id 2, 9), Monitor (id 4, 10).

### Langkah 3: Melihat Struktur DataFrame

In [12]:
df.printSchema()

root
 |-- id_transaksi: long (nullable = true)
 |-- produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- jumlah: long (nullable = true)



> **Interpretasi**
>
> Tipe data ditentukan otomatis oleh Spark (*schema inference*):
>
> | Kolom | Tipe |
> |---|---|
> | `id_transaksi` | long |
> | `produk` | string |
> | `kategori` | string |
> | `harga` | long |
> | `jumlah` | long |
>
> Kolom `harga` dan `jumlah` bertipe numerik sehingga dapat digunakan dalam perhitungan (`harga × jumlah`).

### Langkah 4: Menghitung Jumlah Baris

In [13]:
print("Jumlah transaksi:", df.count())

Jumlah transaksi: 10


> **Interpretasi**
>
> - `count()` (action) menghitung semua baris **10 transaksi**.

### Langkah 5: Memilih Kolom Tertentu
Tampilkan kolom produk, kategori, dan harga.

In [14]:
df.select(
    "produk",
    "kategori",
    "harga"
).show()

+---------+----------+-------+
|   produk|  kategori|  harga|
+---------+----------+-------+
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
| Keyboard| Aksesoris| 300000|
|  Monitor|Elektronik|2500000|
|Flashdisk| Aksesoris|  80000|
|  Printer|Elektronik|1800000|
|  Headset| Aksesoris| 250000|
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
|  Monitor|Elektronik|2500000|
+---------+----------+-------+



> **Interpretasi**
>
> - `select()` hanya mengambil 3 kolom: `produk`, `kategori`, `harga`.
> - Jumlah baris tetap 10, sedangkan kolom `id_transaksi` dan `jumlah` tidak ditampilkan.

### Langkah 6: Melakukan Filtering
Tampilkan transaksi dengan harga produk lebih dari Rp1.000.000.

In [15]:
df.filter(
    df.harga > 1000000
).show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



> **Interpretasi**
>
> - Terdapat **5 transaksi**: Laptop (id 1, 8), Monitor (id 4, 10), dan Printer (id 6).
> - Semuanya berkategori **Elektronik**. Seluruh produk Aksesoris memiliki harga di bawah Rp1.000.000 sehingga tersaring.

### Langkah 7: Membuat Kolom Baru
Tambahkan kolom total transaksi dengan rumus:

$$Total = Harga \times Jumlah$$

In [16]:
from pyspark.sql.functions import col

df_total = df.withColumn(
    "total",
    col("harga") * col("jumlah")
)

df_total.show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



> **Interpretasi**
>
> `withColumn()` menambah kolom `total`. Hasilnya disimpan di `df_total`, sedangkan `df` tidak berubah (DataFrame bersifat *immutable*).
>
> | id | Produk | Perhitungan | Total |
> |:---:|---|---|---:|
> | 1 | Laptop | 7.500.000 × 2 | 15.000.000 |
> | 2 | Mouse | 150.000 × 5 | 750.000 |
> | 3 | Keyboard | 300.000 × 3 | 900.000 |
> | 4 | Monitor | 2.500.000 × 2 | 5.000.000 |
> | 5 | Flashdisk | 80.000 × 10 | 800.000 |
> | 6 | Printer | 1.800.000 × 1 | 1.800.000 |
> | 7 | Headset | 250.000 × 4 | 1.000.000 |
> | 8 | Laptop | 7.500.000 × 1 | 7.500.000 |
> | 9 | Mouse | 150.000 × 8 | 1.200.000 |
> | 10 | Monitor | 2.500.000 × 3 | 7.500.000 |

### Langkah 8: Mengurutkan Data
Urutkan transaksi berdasarkan total transaksi dari terbesar ke terkecil.

In [17]:
df_total.orderBy(
    col("total").desc()
).show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
+------------+---------+----------+-------+------+--------+



> **Interpretasi**
>
> - Transaksi **terbesar**: Laptop id 1 = Rp15.000.000.
> - Transaksi **terkecil**: Mouse id 2 = Rp750.000.
> - Laptop id 8 dan Monitor id 10 sama-sama Rp7.500.000, sehingga urutan keduanya dapat tertukar.


## E.3. Agregasi Data Menggunakan DataFrame API

**Tujuan:** Menganalisis data menggunakan operasi agregasi: total pendapatan per kategori, jumlah transaksi, dan rata-rata nilai transaksi.

### Langkah 1: Total Pendapatan Setiap Kategori

In [18]:
from pyspark.sql.functions import sum

df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan")
    ).show()

+----------+----------------+
|  kategori|total_pendapatan|
+----------+----------------+
|Elektronik|        36800000|
| Aksesoris|         4650000|
+----------+----------------+



> **Interpretasi**
>
> | Kategori | Total Pendapatan |
> |---|---:|
> | Elektronik | 36.800.000 |
> | Aksesoris | 4.650.000 |
>
> Pendapatan kategori Elektronik sekitar **8 kali lipat** pendapatan kategori Aksesoris.

### Langkah 2: Jumlah Transaksi Setiap Kategori

In [19]:
from pyspark.sql.functions import count

df_total.groupBy("kategori") \
    .agg(
        count("*").alias("jumlah_transaksi")
    ).show()

+----------+----------------+
|  kategori|jumlah_transaksi|
+----------+----------------+
|Elektronik|               5|
| Aksesoris|               5|
+----------+----------------+



> **Interpretasi**
>
> - Masing-masing kategori punya **5 transaksi**.
> - Jumlah transaksi sama, tetapi pendapatan sangat berbeda karena harga barang Elektronik jauh lebih tinggi.

### Langkah 3: Rata-rata Nilai Transaksi

In [20]:
from pyspark.sql.functions import avg

df_total.groupBy("kategori") \
    .agg(
        avg("total").alias("rata_rata_transaksi")
    ).show()

+----------+-------------------+
|  kategori|rata_rata_transaksi|
+----------+-------------------+
|Elektronik|          7360000.0|
| Aksesoris|           930000.0|
+----------+-------------------+



> **Interpretasi**
>
> | Kategori | Perhitungan | Rata-rata |
> |---|---|---:|
> | Elektronik | 36.800.000 ÷ 5 | 7.360.000 |
> | Aksesoris | 4.650.000 ÷ 5 | 930.000 |

### Langkah 4: Statistik Keseluruhan

In [21]:
df_total.select(
    sum("total").alias("total_penjualan"),
    avg("total").alias("rata_rata_penjualan"),
    count("*").alias("jumlah_transaksi")
).show()

+---------------+-------------------+----------------+
|total_penjualan|rata_rata_penjualan|jumlah_transaksi|
+---------------+-------------------+----------------+
|       41450000|          4145000.0|              10|
+---------------+-------------------+----------------+



> **Interpretasi**
>
> - Tanpa `groupBy()`, agregasi berlaku pada **seluruh data**:
> - **Total penjualan:** Rp41.450.000 (36.800.000 + 4.650.000)
> - **Rata-rata penjualan:** Rp4.145.000 per transaksi
> - **Jumlah transaksi:** 10

### Pertanyaan Analisis E.3

**1. Kategori produk mana yang menghasilkan pendapatan terbesar?**

**Jawaban:** **Elektronik**, Rp36.800.000 (sekitar 88,8% dari total). Kontributor utamanya adalah Laptop dan Monitor.

**2. Berapa total pendapatan dari seluruh transaksi?**

**Jawaban:** **Rp41.450.000** dari 10 transaksi.

**3. Mengapa operasi `groupBy()` diperlukan dalam analisis data?**

**Jawaban:** `groupBy()` mengelompokkan data berdasarkan suatu kolom agar agregasi dihitung **per kelompok**. Tanpanya hanya diperoleh satu angka untuk seluruh data, sehingga perbandingan antar kategori tidak dapat dilakukan.

**4. Apa perbedaan `sum()`, `count()`, dan `avg()`?**

**Jawaban:** `sum()` **menjumlahkan** nilai kolom (hasil: 41.450.000). `count()` **menghitung banyak baris** (hasil: 10). `avg()` menghitung **rata-rata** = sum ÷ count (hasil: 4.145.000).



## E.4. Pemrosesan Data Menggunakan Spark SQL

**Tujuan:** Memproses data dengan perintah SQL pada Apache Spark.

### Langkah 1: Membuat Temporary View

In [22]:
df_total.createOrReplaceTempView(
    "transaksi"
)

> **Interpretasi**
>
> - `df_total` didaftarkan sebagai *temporary view* bernama `transaksi` sehingga dapat di-query seperti tabel SQL.
> - View hanya berlaku selama SparkSession aktif. Tidak terdapat output.

### Langkah 2: Menampilkan Seluruh Data

In [23]:
spark.sql("""
    SELECT *
    FROM transaksi
""").show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



> **Interpretasi**
>
> - Menampilkan semua kolom dan baris (10 baris, 6 kolom), identik dengan `df_total.show()`.

### Langkah 3: Filtering Menggunakan SQL

In [24]:
spark.sql("""
    SELECT produk, kategori, harga, jumlah, total
    FROM transaksi
    WHERE total > 1000000
""").show()

+-------+----------+-------+------+--------+
| produk|  kategori|  harga|jumlah|   total|
+-------+----------+-------+------+--------+
| Laptop|Elektronik|7500000|     2|15000000|
|Monitor|Elektronik|2500000|     2| 5000000|
|Printer|Elektronik|1800000|     1| 1800000|
| Laptop|Elektronik|7500000|     1| 7500000|
|  Mouse| Aksesoris| 150000|     8| 1200000|
|Monitor|Elektronik|2500000|     3| 7500000|
+-------+----------+-------+------+--------+



> **Interpretasi**
>
> - Terdapat **6 transaksi** dengan `total > 1.000.000`.
> - **Headset (total tepat 1.000.000) tidak termasuk** karena kondisi yang digunakan adalah `>`, bukan `>=`.
> - **Mouse id 9** termasuk karena totalnya 1.200.000 (150.000 × 8), meskipun harga satuannya rendah.
> - Hasil ini berbeda dengan filter `harga > 1000000` pada E.2 (5 baris) karena yang difilter adalah **total**, bukan harga satuan.

### Langkah 4: Agregasi Menggunakan SQL

In [25]:
spark.sql("""
    SELECT kategori,
           SUM(total) AS total_pendapatan,
           COUNT(*) AS jumlah_transaksi,
           AVG(total) AS rata_rata_transaksi
    FROM transaksi
    GROUP BY kategori
""").show()

+----------+----------------+----------------+-------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_transaksi|
+----------+----------------+----------------+-------------------+
|Elektronik|        36800000|               5|          7360000.0|
| Aksesoris|         4650000|               5|           930000.0|
+----------+----------------+----------------+-------------------+



> **Interpretasi**
>
> Satu query menghasilkan tiga agregasi sekaligus (pada DataFrame API di E.3 dilakukan secara terpisah):
>
> | Kategori | Total Pendapatan | Jumlah Transaksi | Rata-rata |
> |---|---:|:---:|---:|
> | Elektronik | 36.800.000 | 5 | 7.360.000 |
> | Aksesoris | 4.650.000 | 5 | 930.000 |

### Langkah 5: Mengurutkan Hasil Query

In [26]:
spark.sql("""
    SELECT produk,
           SUM(total) AS total_penjualan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_penjualan DESC
""").show()

+---------+---------------+
|   produk|total_penjualan|
+---------+---------------+
|   Laptop|       22500000|
|  Monitor|       12500000|
|    Mouse|        1950000|
|  Printer|        1800000|
|  Headset|        1000000|
| Keyboard|         900000|
|Flashdisk|         800000|
+---------+---------------+



> **Interpretasi**
>
> Penjualan dijumlahkan per **produk**, diurutkan dari terbesar:
>
> | # | Produk | Total Penjualan |
> |:---:|---|---:|
> | 1 | Laptop | 22.500.000 |
> | 2 | Monitor | 12.500.000 |
> | 3 | Mouse | 1.950.000 |
> | 4 | Printer | 1.800.000 |
> | 5 | Headset | 1.000.000 |
> | 6 | Keyboard | 900.000 |
> | 7 | Flashdisk | 800.000 |
>
> **Laptop** merupakan produk dengan penjualan tertinggi (sekitar 54% dari total pendapatan).

### Tambahan (opsional): Membuktikan Hasil SQL = DataFrame API


In [27]:
from pyspark.sql.functions import sum, count, avg

hasil_df = df_total.groupBy("kategori").agg(
    sum("total").alias("total_pendapatan"),
    count("*").alias("jumlah_transaksi"),
    avg("total").alias("rata_rata_transaksi")
)

hasil_sql = spark.sql("""
    SELECT kategori,
           SUM(total) AS total_pendapatan,
           COUNT(*) AS jumlah_transaksi,
           AVG(total) AS rata_rata_transaksi
    FROM transaksi
    GROUP BY kategori
""")

# Selisih baris di kedua arah: 0 berarti hasil identik
print("Baris di DataFrame API tapi tidak di SQL:", hasil_df.exceptAll(hasil_sql).count())
print("Baris di SQL tapi tidak di DataFrame API:", hasil_sql.exceptAll(hasil_df).count())

Baris di DataFrame API tapi tidak di SQL: 0
Baris di SQL tapi tidak di DataFrame API: 0


> **Interpretasi**
>
> - Kedua selisih bernilai **0**, sehingga hasil DataFrame API dan Spark SQL **identik**.

### Pertanyaan E.4

**1. Bandingkan hasil Spark SQL dengan DataFrame API pada bagian E.3. Apakah hasilnya sama?**

**Jawaban:** **Ya, sama.** Total pendapatan (Elektronik 36.800.000; Aksesoris 4.650.000), jumlah transaksi (5 dan 5), dan rata-rata (7.360.000 dan 930.000) identik. Keduanya dioptimalkan oleh engine yang sama (Catalyst) sehingga menghasilkan *execution plan* yang setara. Perbedaannya hanya terletak pada gaya penulisan: SQL sesuai bagi pengguna yang terbiasa dengan basis data, sedangkan DataFrame API sesuai untuk penulisan kode Python.


## F. Eksperimen Lazy Evaluation

**Tujuan:** Mengamati konsep *lazy evaluation* pada Apache Spark.

### Percobaan 1: Mendefinisikan Transformation

In [28]:
df_filter = df.filter(
    df.harga > 1000000
)

print("Transformation selesai")

Transformation selesai


> **Interpretasi**
>
> - Yang tercetak hanya teks `Transformation selesai`, **tanpa menampilkan data**.
> - `filter()` merupakan transformation sehingga Spark baru mencatat rencana dan belum memproses data.

### Percobaan 2: Memanggil Action

In [29]:
df_filter.show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



> **Interpretasi**
>
> - Setelah `show()` (action) dipanggil, data baru tampil: **5 transaksi** dengan `harga > 1.000.000`.
> - Hasil ini sama dengan E.2 Langkah 6. Pada tahap inilah Spark benar-benar menjalankan pemrosesan.

### Percobaan 3: Melihat Execution Plan

In [30]:
df_filter.explain()

== Physical Plan ==
*(1) Filter (isnotnull(harga#3L) AND (harga#3L > 1000000))
+- *(1) Scan ExistingRDD[id_transaksi#0L,produk#1,kategori#2,harga#3L,jumlah#4L]




> **Interpretasi**
>
> - `explain()` menampilkan *physical plan*, dibaca dari **bawah ke atas**:
> - **Scan ExistingRDD**: membaca sumber data (DataFrame yang dibuat dari data Python).
> - **Filter**: kondisi `harga > 1000000` yang ditulis pada kode, ditambah `isnotnull(harga)` yang ditambahkan otomatis oleh optimizer.
> - Hal ini membuktikan bahwa Spark menyusun rencana terlebih dahulu sebelum menjalankan pemrosesan.

### Pertanyaan F

**1. Mengapa Spark menggunakan lazy evaluation?**

**Jawaban:** Agar Spark dapat melihat **seluruh rangkaian operasi** dulu, lalu menyusun rencana paling efisien, tanpa memproses data yang pada akhirnya tidak diperlukan.

**2. Apa keuntungan lazy evaluation dalam pemrosesan Big Data?**

**Jawaban:** **Optimasi menyeluruh** (misalnya filter lebih awal agar data yang diproses lebih sedikit), **hemat resource** (CPU, memori, I/O), **menghindari hasil antara yang besar**, dan **fault tolerance** karena *lineage* tersimpan sehingga partisi yang hilang dapat dihitung ulang.

**3. Apa fungsi execution plan dalam Apache Spark?**

**Jawaban:** Berfungsi sebagai **rencana kerja** yang dioptimalkan (oleh Catalyst Optimizer) sebelum dijalankan di executor, sekaligus sarana untuk memahami dan menganalisis performa query melalui `explain()`.

**4. Apa perbedaan antara mendefinisikan transformation dan mengeksekusi action?**

**Jawaban:** **Mendefinisikan transformation** hanya mencatat rencana (belum ada pemrosesan dan output; contoh: `df_filter = df.filter(...)` hanya mencetak "Transformation selesai"). **Mengeksekusi action** menjalankan seluruh rencana dan menghasilkan data/nilai (contoh: `df_filter.show()` baru menampilkan tabel).

---

## G. Tugas Praktikum 6

Tugas berikut dikerjakan menggunakan dataset transaksi yang telah dibuat pada bagian E.2 (DataFrame `df` dan `df_total`, serta *temporary view* `transaksi`).

### Tugas 1: Analisis Produk
Tampilkan lima produk dengan total pendapatan tertinggi. Gunakan DataFrame API.

In [31]:
from pyspark.sql.functions import col, sum

df_total.groupBy("produk") \
    .agg(
        sum("total").alias("total_pendapatan")
    ) \
    .orderBy(col("total_pendapatan").desc()) \
    .limit(5) \
    .show()

+-------+----------------+
| produk|total_pendapatan|
+-------+----------------+
| Laptop|        22500000|
|Monitor|        12500000|
|  Mouse|         1950000|
|Printer|         1800000|
|Headset|         1000000|
+-------+----------------+



> **Interpretasi**
>
> Alur kode: `groupBy("produk")` mengelompokkan transaksi per produk, `agg(sum("total"))` menjumlahkan pendapatan tiap produk, `orderBy(...desc())` mengurutkan dari terbesar, dan `limit(5)` mengambil lima baris teratas.
>
> | Peringkat | Produk | Total Pendapatan |
> |:---:|---|---:|
> | 1 | Laptop | 22.500.000 |
> | 2 | Monitor | 12.500.000 |
> | 3 | Mouse | 1.950.000 |
> | 4 | Printer | 1.800.000 |
> | 5 | Headset | 1.000.000 |
>
> - Laptop dan Monitor menyumbang 35.000.000 dari total 41.450.000, atau sekitar 84,4% pendapatan.
> - Keyboard (900.000) dan Flashdisk (800.000) tidak masuk lima besar.
> - Produk yang sama pada transaksi berbeda dijumlahkan lebih dahulu (Laptop = 15.000.000 + 7.500.000), sehingga peringkat berdasarkan **produk**, bukan berdasarkan transaksi tunggal.

### Tugas 2: Analisis Kategori
Hitung total pendapatan setiap kategori, jumlah transaksi setiap kategori, dan rata-rata jumlah produk yang terjual setiap kategori.

In [32]:
from pyspark.sql.functions import sum, count, avg

df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan"),
        count("*").alias("jumlah_transaksi"),
        avg("jumlah").alias("rata_rata_jumlah_terjual")
    ).show()

+----------+----------------+----------------+------------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_jumlah_terjual|
+----------+----------------+----------------+------------------------+
|Elektronik|        36800000|               5|                     1.8|
| Aksesoris|         4650000|               5|                     6.0|
+----------+----------------+----------------+------------------------+



> **Interpretasi**
>
> | Kategori | Total Pendapatan | Jumlah Transaksi | Rata-rata Jumlah Terjual |
> |---|---:|:---:|:---:|
> | Elektronik | 36.800.000 | 5 | 1,8 unit per transaksi |
> | Aksesoris | 4.650.000 | 5 | 6,0 unit per transaksi |
>
> - Rata-rata jumlah terjual dihitung dari kolom `jumlah` (Elektronik: (2 + 2 + 1 + 1 + 3) / 5 = 1,8; Aksesoris: (5 + 3 + 10 + 4 + 8) / 5 = 6,0).
> - Terdapat pola berlawanan: Elektronik bernilai besar tetapi dibeli sedikit per transaksi, sedangkan Aksesoris bernilai kecil tetapi dibeli dalam jumlah banyak. Satu transaksi Aksesoris rata-rata memuat unit 3,3 kali lebih banyak, tetapi nilainya hanya sekitar 12,6% dari transaksi Elektronik.
> - Ketiga agregasi dihitung dalam satu `agg()`, sehingga data hanya dikelompokkan sekali.

### Tugas 3: Filtering Data
Tampilkan transaksi dengan harga produk lebih dari Rp100.000 dan jumlah produk yang dibeli minimal 3 unit.

In [33]:
from pyspark.sql.functions import col

df.filter(
    (col("harga") > 100000) & (col("jumlah") >= 3)
).show()

+------------+--------+----------+-------+------+
|id_transaksi|  produk|  kategori|  harga|jumlah|
+------------+--------+----------+-------+------+
|           2|   Mouse| Aksesoris| 150000|     5|
|           3|Keyboard| Aksesoris| 300000|     3|
|           7| Headset| Aksesoris| 250000|     4|
|           9|   Mouse| Aksesoris| 150000|     8|
|          10| Monitor|Elektronik|2500000|     3|
+------------+--------+----------+-------+------+



> **Interpretasi**
>
> - Dua kondisi digabung dengan operator `&` (dan), sehingga baris harus memenuhi **keduanya**. Setiap kondisi wajib diberi tanda kurung karena `&` memiliki prioritas lebih tinggi daripada operator pembanding pada Python.
> - Terdapat **5 transaksi** yang lolos: id 2 (Mouse, 5 unit), id 3 (Keyboard, 3 unit), id 7 (Headset, 4 unit), id 9 (Mouse, 8 unit), dan id 10 (Monitor, 3 unit).
> - Transaksi yang gugur karena jumlah di bawah 3 unit: Laptop id 1 (2 unit), Monitor id 4 (2 unit), Printer id 6 (1 unit), dan Laptop id 8 (1 unit). Flashdisk id 5 gugur karena harga Rp80.000 tidak lebih dari Rp100.000 meskipun dibeli 10 unit.
> - Keyboard id 3 lolos karena syarat jumlah bersifat *minimal* (`>= 3`), sehingga tepat 3 unit masih memenuhi.

### Tugas 4: Spark SQL
Buat query SQL untuk menampilkan nama produk dan total pendapatan setiap produk, diurutkan berdasarkan pendapatan terbesar.

In [34]:
spark.sql("""
    SELECT produk,
           SUM(total) AS total_pendapatan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_pendapatan DESC
""").show()

+---------+----------------+
|   produk|total_pendapatan|
+---------+----------------+
|   Laptop|        22500000|
|  Monitor|        12500000|
|    Mouse|         1950000|
|  Printer|         1800000|
|  Headset|         1000000|
| Keyboard|          900000|
|Flashdisk|          800000|
+---------+----------------+



> **Interpretasi**
>
> - Query ini memakai *temporary view* `transaksi` dari E.4. `GROUP BY produk` mengelompokkan data, `SUM(total)` menjumlahkan pendapatan, dan `ORDER BY total_pendapatan DESC` mengurutkan dari terbesar.
> - Hasilnya berisi **7 produk** dan urutannya sama dengan Tugas 1 dan E.4 Langkah 5: Laptop (22.500.000), Monitor (12.500.000), Mouse (1.950.000), Printer (1.800.000), Headset (1.000.000), Keyboard (900.000), dan Flashdisk (800.000).
> - Perbedaannya dengan Tugas 1 hanya pada cakupan: Tugas 1 dibatasi lima baris dengan `limit(5)`, sedangkan query ini menampilkan seluruh produk.

### Tugas 5: Analisis Konsep
Jelaskan perbedaan RDD dan DataFrame serta berikan contoh kondisi penggunaan masing-masing.

**Jawaban:**

RDD (*Resilient Distributed Dataset*) adalah abstraksi tingkat rendah berupa kumpulan objek terdistribusi tanpa skema yang dipahami Spark. DataFrame adalah abstraksi tingkat tinggi berupa data terdistribusi berbentuk baris dan kolom dengan skema, sehingga Spark mengetahui nama dan tipe tiap kolom.

| Aspek | RDD | DataFrame |
|---|---|---|
| Struktur data | Kumpulan objek tanpa skema | Baris dan kolom dengan skema |
| Tingkat abstraksi | Rendah: pengguna menulis sendiri logika pemrosesan | Tinggi: API deklaratif (`select`, `filter`, `groupBy`) dan SQL |
| Optimasi | Tidak ada optimasi otomatis | Dioptimalkan oleh Catalyst dan Tungsten |
| Eksekusi fungsi | *Lambda* Python berjalan di proses Python dengan serialisasi data | Fungsi bawaan berjalan di JVM |
| Dukungan SQL | Tidak ada | Ada, melalui *temporary view* dan `spark.sql()` |
| Kemudahan penulisan | Kode lebih panjang | Kode lebih ringkas dan mudah dibaca |

**Contoh kondisi penggunaan RDD:**
- Data **tidak terstruktur atau semi-terstruktur** yang tidak cocok dibentuk menjadi tabel, misalnya teks mentah, log yang formatnya tidak beraturan, atau objek biner.
- Memerlukan **kontrol tingkat rendah** atas partisi dan logika pemrosesan, misalnya algoritma khusus yang tidak dapat dinyatakan dengan operasi DataFrame.
- Memerlukan operasi fungsional per elemen seperti `map()`, `flatMap()`, dan `reduce()` pada data yang bukan baris-kolom (contohnya, Percobaan RDD pada E.1).

**Contoh kondisi penggunaan DataFrame:**
- Data **terstruktur atau tabular**, misalnya transaksi penjualan, hasil ekspor basis data, atau berkas CSV dan Parquet.
- Analisis yang memerlukan **agregasi, filter, join, dan pengurutan**, seperti pada E.2 sampai E.4.
- Pekerjaan yang membutuhkan **kinerja optimal** dan **kueri SQL**, karena DataFrame dioptimalkan otomatis dan dapat diakses melalui SQL.

Sebagai praktik umum, DataFrame dipilih untuk data tabular karena lebih ringkas dan lebih cepat, sedangkan RDD dipakai hanya bila DataFrame tidak mencukupi.

#### Tambahan (opsional): Membandingkan Kode RDD dan DataFrame


In [35]:
# Versi RDD: baris diubah menjadi pasangan (kategori, total), lalu dijumlahkan per kunci
hasil_rdd = df_total.rdd \
    .map(lambda r: (r["kategori"], r["total"])) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()

print("Hasil RDD       :", sorted(hasil_rdd))

# Versi DataFrame: satu baris deklaratif
hasil_df_list = [(r["kategori"], r["total_pendapatan"]) for r in
                 df_total.groupBy("kategori").agg(sum("total").alias("total_pendapatan")).collect()]

print("Hasil DataFrame :", sorted(hasil_df_list))

Hasil RDD       : [('Aksesoris', 4650000), ('Elektronik', 36800000)]
Hasil DataFrame : [('Aksesoris', 4650000), ('Elektronik', 36800000)]


> **Interpretasi**
>
> - Kedua pendekatan menghasilkan nilai yang sama: Aksesoris 4.650.000 dan Elektronik 36.800.000.
> - Pada versi RDD, pengguna harus menentukan sendiri cara membentuk pasangan kunci-nilai dan cara menggabungkannya (`reduceByKey`). Pada versi DataFrame, cukup menyatakan *apa* yang diinginkan (`groupBy` dan `sum`), dan Spark menentukan cara eksekusinya.
> - Fungsi `collect()` dan `sorted()` hanya dipakai agar hasil dapat dibandingkan secara langsung pada data sekecil ini.



## H. Pertanyaan Evaluasi

Jawablah pertanyaan berikut berdasarkan hasil praktikum.

### Jawaban Pertanyaan Evaluasi

**1. Apa yang dimaksud dengan Apache Spark dan apa keunggulannya dalam pemrosesan Big Data?**

**Jawaban:** Apache Spark adalah *framework* komputasi terdistribusi yang digunakan untuk memproses data dalam jumlah besar. Spark mendukung *batch processing*, *interactive query* dengan Spark SQL, *streaming* dengan Structured Streaming, *machine learning* dengan MLlib, dan *graph processing* dengan GraphX. Keunggulannya: (1) **pemrosesan di memori** (*in-memory computing*) yang mempercepat komputasi; (2) **pemrosesan paralel dan terdistribusi** karena data dibagi menjadi partisi yang diproses serentak; (3) **dukungan banyak bahasa** (Python, Scala, Java, dan R); dan (4) **API tingkat tinggi** seperti DataFrame dan SQL yang menyederhanakan penulisan program. Keunggulan ini terlihat pada praktikum: seluruh pemrosesan dapat ditulis dengan kode singkat, dan rencana eksekusi dioptimalkan otomatis.

**2. Jelaskan perbedaan *transformation* dan *action* pada RDD. Berikan masing-masing dua contoh.**

**Jawaban:** *Transformation* adalah operasi yang menghasilkan RDD baru dari RDD sebelumnya dan bersifat *lazy*, sehingga hanya mencatat rencana tanpa memproses data. Contohnya `map()` (mengalikan tiap elemen dengan 2 pada E.1 Langkah 3) dan `filter()` (mengambil elemen lebih besar dari 50 pada E.1 Langkah 4). *Action* adalah operasi yang memicu eksekusi dan menghasilkan nilai atau mengembalikan hasil ke driver. Contohnya `collect()` (mengambil seluruh isi RDD) dan `count()` (menghitung jumlah elemen, bernilai 10 pada praktikum). Perbedaan utamanya terletak pada waktu eksekusi dan jenis hasil: *transformation* mengembalikan rencana, *action* mengembalikan hasil.

**3. Mengapa Apache Spark menerapkan *lazy evaluation*?**

**Jawaban:** Agar Spark dapat melihat seluruh rangkaian operasi sebelum menjalankannya, lalu menyusun rencana eksekusi yang paling efisien. Dengan cara ini Spark dapat menggabungkan beberapa langkah dalam satu tahap, melakukan filter lebih awal, dan menghindari pemrosesan atau hasil antara yang tidak diperlukan. Pada Percobaan F terlihat bahwa `df.filter(...)` hanya mencetak `Transformation selesai` tanpa data, dan data baru muncul setelah `show()` dipanggil. Rencana yang disusun dapat dilihat melalui `explain()`.

**4. Apa perbedaan RDD dan DataFrame?**

**Jawaban:** RDD adalah kumpulan objek terdistribusi tanpa skema dengan abstraksi tingkat rendah, tanpa optimasi otomatis, dan logikanya ditulis manual (misalnya dengan fungsi *lambda*). DataFrame adalah data terdistribusi berbentuk baris dan kolom dengan skema (nama dan tipe kolom), API deklaratif, dukungan SQL, dan dioptimalkan oleh Catalyst. RDD sesuai untuk data tidak terstruktur dan kontrol tingkat rendah, sedangkan DataFrame sesuai untuk data tabular dan analisis terstruktur. Perbandingan lengkap dan contoh kondisi penggunaan terdapat pada Tugas 5.

**5. Mengapa DataFrame lebih mudah digunakan untuk melakukan analisis data terstruktur?**

**Jawaban:** Pertama, DataFrame memiliki **skema**, sehingga kolom dapat dirujuk dengan nama dan tipenya diketahui (lihat `printSchema()`). Kedua, tersedia **operasi tingkat tinggi** seperti `select()`, `filter()`, `groupBy()`, `agg()`, dan `orderBy()` yang menyatakan maksud analisis secara langsung. Ketiga, DataFrame dapat dikueri dengan **SQL**, yang akrab bagi analis. Keempat, **optimasi dilakukan otomatis** oleh Catalyst sehingga pengguna tidak perlu mengatur urutan operasi secara manual. Hal ini terlihat pada Tugas 5: agregasi per kategori memerlukan beberapa langkah dengan RDD, tetapi cukup satu pernyataan `groupBy().agg()` dengan DataFrame.

**6. Apa hubungan antara Spark SQL dan DataFrame API?**

**Jawaban:** Keduanya adalah dua antarmuka untuk mesin pemrosesan yang sama. DataFrame dapat didaftarkan sebagai *temporary view* dengan `createOrReplaceTempView()` sehingga dapat dikueri dengan `spark.sql()`, dan hasil `spark.sql()` sendiri adalah DataFrame yang dapat diproses lanjut dengan DataFrame API. Kueri SQL dan operasi DataFrame sama-sama diterjemahkan oleh Catalyst menjadi rencana eksekusi yang setara. Kesetaraan ini dibuktikan pada E.4: hasil agregasi per kategori melalui SQL dan melalui DataFrame API identik, dengan selisih `exceptAll()` bernilai 0 pada kedua arah.

**7. Bagaimana Apache Spark dapat mempercepat pemrosesan data dibandingkan pendekatan yang banyak melakukan operasi baca dan tulis ke disk?**

**Jawaban:** Pendekatan berbasis disk (seperti Hadoop MapReduce) menulis hasil antara setiap tahap ke disk, lalu membacanya kembali pada tahap berikutnya. Spark mempercepat pemrosesan dengan beberapa cara: (1) menyimpan data dan hasil antara **di memori** sehingga akses jauh lebih cepat daripada disk; (2) merangkai operasi *narrow* seperti `filter` dan `map` dalam **satu tahap** tanpa menulis hasil antara; (3) menerapkan ***lazy evaluation*** dan optimasi rencana eksekusi sehingga pekerjaan yang tidak perlu dihindari; dan (4) memproses data **secara paralel** pada banyak partisi. Keunggulan terbesar tampak pada pekerjaan iteratif yang memakai ulang data berkali-kali, karena data cukup dimuat sekali dan dipakai ulang dari memori. Perlu dicatat bahwa bila memori tidak mencukupi, Spark tetap dapat memakai disk, dan pada data sekecil praktikum ini keunggulan kecepatan tidak terlihat karena *overhead* Spark lebih besar daripada waktu pemrosesan.